In [ ]:
import numpy as np
import tempfile
from pathlib import Path

from numcompute.io import load_csv
from numcompute.preprocessing import StandardScaler, SimpleImputer, OneHotEncoder
from numcompute.sort_search import topk, quickselect, binary_search
from numcompute.rank import rank, percentile
from numcompute.stats import mean, std, histogram, quantile
from numcompute.metrics import accuracy, precision, recall, f1, confusion_matrix, mse
from numcompute.optim import grad, jacobian
from numcompute.pipeline import Pipeline
from numcompute.benchmarking import (
    compare_functions,
    vectorized_sum_of_squares,
    loop_sum_of_squares,
    format_benchmark_table,
)

ModuleNotFoundError: No module named 'numcompute'

In [ ]:
csv_text = "1,10,\n2,,0\n3,30,1\n4,40,0"

with tempfile.NamedTemporaryFile(mode="w+", suffix=".csv", delete=False) as f:
    f.write(csv_text)
    temp_path = f.name

X_raw = load_csv(temp_path, delimiter=",")
print("Loaded CSV:")
print(X_raw)

In [ ]:
X_num = X_raw[:, :2]
y = X_raw[:, 2]

imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X_num)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print("Imputed data:")
print(X_imputed)
print("\nScaled data:")
print(X_scaled)

[[-1. -1.]
 [ 1.  1.]]


In [ ]:
X_cat = np.array([["A"], ["B"], ["A"], ["C"]])
encoder = OneHotEncoder()
X_encoded = encoder.fit_transform(X_cat)

print("One-hot encoded:")
print(X_encoded)

In [ ]:
values = np.array([10, 30, 20, 20, 50, 40])

print("Top-3 largest:")
print(topk(values, k=3, largest=True, return_indices=False))

print("\nQuickselect k=2 (3rd smallest):")
print(quickselect(values, 2))

sorted_vals = np.sort(values)
print("\nSorted values:")
print(sorted_vals)

print("\nBinary search for 30:")
print(binary_search(sorted_vals, 30))

print("\nRanks (average ties):")
print(rank(values, method="average"))

print("\nPercentile 75:")
print(percentile(values, 75))

In [ ]:
print("Mean:", mean(values))
print("Std:", std(values))
print("Quantile 0.5:", quantile(values, 0.5))

counts, bins = histogram(values, bins=4)
print("\nHistogram counts:", counts)
print("Histogram bins:", bins)

In [ ]:
y_true = np.array([0, 1, 1, 0, 1])
y_pred = np.array([0, 1, 0, 0, 1])

print("Accuracy:", accuracy(y_true, y_pred))
print("Precision:", precision(y_true, y_pred))
print("Recall:", recall(y_true, y_pred))
print("F1:", f1(y_true, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

y_true_reg = np.array([1.0, 2.0, 3.0])
y_pred_reg = np.array([1.2, 1.9, 2.8])
print("MSE:", mse(y_true_reg, y_pred_reg))

In [ ]:
def f(x):
    return x[0] ** 2 + x[1] ** 2

def F(x):
    return np.array([
        x[0] + x[1],
        x[0] * x[1],
    ])

x = np.array([3.0, 4.0])

print("Gradient:", grad(f, x))
print("Jacobian:")
print(jacobian(F, x))

In [ ]:
class DummyModel:
    def fit(self, X, y=None):
        self.threshold_ = np.mean(X[:, 0])
        return self

    def predict(self, X):
        return (X[:, 0] > self.threshold_).astype(int)

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", DummyModel()),
])

pipe.fit(X_num, y=np.array([0, 0, 1, 1]))
preds = pipe.predict(X_num)

print("Pipeline predictions:")
print(preds)

In [ ]:
x = np.random.rand(1_000_000)

result = compare_functions(
    vectorized_sum_of_squares,
    loop_sum_of_squares,
    x,
    repeat=5,
    warmup=1,
)

print(format_benchmark_table([result["first"], result["second"]]))
print("Speedup:", f"{result['speedup_vs_slow']:.2f}x")